# Shot-noise frozen-seed fix: before vs after

Demonstrates that the decoder's simulated shot noise now varies across calls instead of being frozen at a fixed default seed.

In [ ]:
import jax
import numpy as np
import matplotlib.pyplot as plt

from qugen.main.generator.continuous_qgan_model_handler import ContinuousQGANModelHandler

model = ContinuousQGANModelHandler()
model.build(
    model_name="demo",
    data_set="toy",
    n_qubits=5,
    circuit_depth=2,
    measurement_scheme="comp_basis_probs",
    decoding_scheme="FRQI",
    discriminator_name="continuous_fully_connected",
    shots=256,  # must be set for this fix to have any visible effect
    save_artifacts=False,
)

## One fixed generator output

So any difference we see below comes only from how the decoder is called, not from the generator changing.

In [ ]:
model.generator_weights = model.gen_init_sample_fn(
    key=jax.random.PRNGKey(1), shape=model.params_shape, scale=0.5, shift=0.0
)
v_qnode = jax.vmap(lambda inpt: model.generator(inpt, model.generator_weights))
noise = model.noise_sample_fn(key=jax.random.PRNGKey(2), shape=(4, *model.noise_shape), scale=0.3, shift=0.0)
measurement_outputs = model.standardize_pennylane_output(v_qnode(noise))

## Before vs after

BEFORE: no `rng_key` passed in -- this is exactly what the old code did, silently reusing the same hardcoded default seed every call.

AFTER: a fresh key passed in each call -- this is what the training loop does now.

In [ ]:
old1 = model.decoder(measurement_outputs)
old2 = model.decoder(measurement_outputs)
print("BEFORE — two calls identical:", bool(np.allclose(old1, old2)))

new1 = model.decoder(measurement_outputs, rng_key=jax.random.PRNGKey(10))
new2 = model.decoder(measurement_outputs, rng_key=jax.random.PRNGKey(20))
print("AFTER — two calls identical:", bool(np.allclose(new1, new2)))
print("AFTER — max difference:", float(np.max(np.abs(new1 - new2))))

## See it as images

Expect BEFORE's two images to be pixel-identical, and AFTER's two images to visibly differ.

In [ ]:
imgs = [np.asarray(x).reshape(4, 8, 8)[0] for x in [old1, old2, new1, new2]]
fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for ax, img, title in zip(axes, imgs, ["BEFORE call 1", "BEFORE call 2", "AFTER call 1", "AFTER call 2"]):
    ax.imshow(np.clip(img, 0, 1), cmap="gray", vmin=0, vmax=1)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.show()